In [17]:
from kafka import KafkaConsumer
import json
import matplotlib.pyplot as plt
import time

In [18]:
# Initialize data storage for plotting
timestamps = []
water_temperatures = []
ph_levels = []
turbidities = []
dissolved_oxygen_levels = []

In [19]:
# Kafka configuration
def initialize_consumer():
    kafka_topic = "water_quality"
    kafka_bootstrap_servers = ["localhost:9092"]

    # Create Kafka consumer
    consumer = KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='latest',
        enable_auto_commit=True
    )
    return consumer

In [20]:
# Receive all published messages and update plot
def update_plot(consumer):
    try:
        for message in consumer:
            # Parse the message
            sensor_data = message.value
            print(f"Received: {sensor_data}")

            # Update data storage
            timestamps.append(sensor_data['timestamp'])
            water_temperatures.append(sensor_data['water_temperature'])
            ph_levels.append(sensor_data['ph_level'])
            turbidities.append(sensor_data['turbidity'])
            dissolved_oxygen_levels.append(sensor_data['dissolved_oxygen'])

            # Keep only the last 100 entries for plotting
            if len(timestamps) > 100:
                timestamps.pop(0)
                water_temperatures.pop(0)
                ph_levels.pop(0)
                turbidities.pop(0)
                dissolved_oxygen_levels.pop(0)

            # Clear the current axes and redraw the plots
            plt.figure(figsize=(10, 8))

            plt.subplot(2, 2, 1)
            plt.plot(timestamps, water_temperatures, label="Water Temperature", color="blue")
            plt.title("Water Temperature")
            plt.ylabel("°C")

            plt.subplot(2, 2, 2)
            plt.plot(timestamps, ph_levels, label="pH Level", color="green")
            plt.title("pH Level")
            plt.ylabel("pH")

            plt.subplot(2, 2, 3)
            plt.plot(timestamps, turbidities, label="Turbidity", color="orange")
            plt.title("Turbidity")
            plt.ylabel("NTU")

            plt.subplot(2, 2, 4)
            plt.plot(timestamps, dissolved_oxygen_levels, label="Dissolved Oxygen", color="red")
            plt.title("Dissolved Oxygen")
            plt.ylabel("mg/L")

            plt.tight_layout()

            # Save the plot as an image
            plt.savefig(f"water_quality_plot.png")
            plt.close()

            break  # Process one message at a time
    except KeyboardInterrupt:
        print("Stopped consuming messages.")
        consumer.close()

# 🤖 Task 3: ML

## Funciones para realiar la prediccion

In [21]:
import joblib
import pandas as pd
MODEL_PATH = "aeration_classifier.joblib"

def load_prediction_model(model_uri):
    """Carga el modelo desde el archivo local."""
    try:
        model = joblib.load(model_uri)
        print(f":D Modelo cargado exitosamente desde {model_uri}")
        return model
    except Exception as e:
        print(f":( Error al cargar el modelo: {e}")
        return None

def perform_inference(model, sensor_data):
    """
    Realiza la predicción y calcula las probabilidades basadas en los 4 parámetros.
    Retorna: (prediccion, probabilidades)
    """
    try:
        # Extraemos los datos del diccionario
        input_data = [
            sensor_data["water_temperature"], 
            sensor_data["ph_level"], 
            sensor_data["turbidity"], 
            sensor_data["dissolved_oxygen"]
        ]
        
        # 1. Predicción de la clase (ej: 0 o 1)
        prediction = model.predict([input_data])[0]
        
        # 2. Probabilidades de las clases (ej: [0.15, 0.85])
        probabilities = model.predict_proba([input_data])[0]
        
        return prediction, probabilities

    except Exception as e:
        print(f"[Error] durante la inferencia: {e}")
        return None, None

## Productor de la probabilidad de prediccion
Debe publicar:
- Resultados de inferencia
- probabilidad de pertenencia a la clase
- Imprimir por pantalla

In [22]:
from kafka import KafkaProducer

In [23]:
# Definimos un nuevo Topic donde se publicaran los resultados de inferencia
pred_topic = "prediction_topic"
kafka_bootstrap_servers = ["localhost:9092"] 

producer = KafkaProducer(
    bootstrap_servers=kafka_bootstrap_servers,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
print(f"Producing messages to Kafka topic '{pred_topic}'...")



Producing messages to Kafka topic 'prediction_topic'...


# Main
En este bloque de codigo ocurre todo el flujo de datos: 
1. El consumer recibe los datos y los llama al plot (como se hacia originalmente)
2. El consumer llama al modelo y realiza una prediccion
3. La prediccion se envia al nuevo topic

In [ ]:
from datetime import datetime

consumer = initialize_consumer()
print("Subscribed to Kafka topic 'water_quality'.")

model = load_prediction_model(MODEL_PATH)

try:
    while True:
        # Mantener la actualización del plot
        update_plot(consumer)

        try:
            for message in consumer:
                sensor_data = message.value
                time_stp = sensor_data["timestamp"]

                # Convertir timestamp a formato legible
                dt = datetime.fromtimestamp(time_stp)
                formatted_time = dt.strftime("%Y-%m-%d %H:%M:%S")

                # Inferencia del modelo
                prediction, probas = perform_inference(model, sensor_data)

                # Etiqueta legible
                label = "NO AERATION" if prediction == 0 else "AERATION"

                # Print limpio
                print(
                    f"[{formatted_time}] \t| "
                    f"Prediction: {label} ({prediction}) \t| "
                    f"Probabilities -> "
                    f"NO_AERATION: {probas[0]:.2%}, "
                    f"AERATION: {probas[1]:.2%}"
                )

                # Envío a Kafka
                send_data = {
                    "timestamp": time_stp,
                    "prediction": int(prediction),
                    "proba_no_areation": float(probas[0]),
                    "proba_areation": float(probas[1])
                }

                producer.send(pred_topic, send_data)

        except KeyboardInterrupt:
            print("Stopped consuming messages.")
            consumer.close()
            break

except KeyboardInterrupt:
    print("Stopped inference.")
    consumer.close()

Subscribed to Kafka topic 'water_quality'.
:D Modelo cargado exitosamente desde aeration_classifier.joblib


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Received: {'timestamp': 1773917724, 'water_temperature': 26.664650712695806, 'ph_level': 7.99646844193013, 'turbidity': 24.63, 'dissolved_oxygen': 11.48}
[2026-03-19 10:55:25] 	| Prediction: NO AERATION (0) 	| Probabilities -> NO_AERATION: 100.00%, AERATION: 0.00%
[2026-03-19 10:55:26] 	| Prediction: NO AERATION (0) 	| Probabilities -> NO_AERATION: 100.00%, AERATION: 0.00%
[2026-03-19 10:55:27] 	| Prediction: AERATION (1) 	| Probabilities -> NO_AERATION: 0.00%, AERATION: 100.00%
[2026-03-19 10:55:28] 	| Prediction: NO AERATION (0) 	| Probabilities -> NO_AERATION: 100.00%, AERATION: 0.00%
[2026-03-19 10:55:29] 	| Prediction: NO AERATION (0) 	| Probabilities -> NO_AERATION: 100.00%, AERATION: 0.00%
[2026-03-19 10:55:30] 	| Prediction: AERATION (1) 	| Probabilities -> NO_AERATION: 0.00%, AERATION: 100.00%
[2026-03-19 10:55:31] 	| Prediction: AERATION (1) 	| Probabilities -> NO_AERATION: 0.00%, AERATION: 100.00%
[2026-03-19 10:55:32] 	| Prediction: NO AERATION (0) 	| Probabilities -> NO_AE